# Online Retail Dataset — Data Cleaning

**Goal:** Clean the raw transactional data so it's ready for SQL analysis and visualization.

**Dataset:** UK-based online retailer, transactions from Dec 2010 to Dec 2011.

**Issues identified during initial inspection:**
- Missing product descriptions (1,454 rows)
- Missing CustomerID (135,080 rows — ~25% of data)
- Duplicate rows (5,268)
- Cancelled orders mixed in (InvoiceNo starting with 'C')
- Negative quantities (returns/cancellations)
- Zero or negative unit prices (likely data errors, not real transactions)

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/online_retail.csv')
print('Shape:', df.shape)
df.head()

Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Step 1: Initial inspection

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


In [4]:
print('Missing values:')
print(df.isnull().sum())
print()
print('Duplicate rows:', df.duplicated().sum())

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Duplicate rows: 5268


## Step 2: Handle cancellations

Orders with InvoiceNo starting with 'C' are cancellations. These aren't real sales, so we'll separate them out.
We could analyze them separately later (e.g., 'which products get cancelled most'), but for the main revenue analysis we'll exclude them.

In [5]:
df['InvoiceNo'] = df['InvoiceNo'].astype(str)

cancellations = df[df['InvoiceNo'].str.startswith('C')]
print('Cancelled order rows:', len(cancellations))

# Save cancellations separately in case we want to analyze returns later
cancellations.to_csv('../data/cancellations.csv', index=False)

# Remove cancellations from main dataset
df = df[~df['InvoiceNo'].str.startswith('C')]
print('Remaining rows after removing cancellations:', len(df))

Cancelled order rows: 9288
Remaining rows after removing cancellations: 532621


## Step 3: Remove invalid transactions

Negative quantities and zero/negative prices that remain after removing cancellations are likely data errors (not legitimate sales), so we drop them.

In [6]:
print('Negative quantity rows remaining:', (df['Quantity'] <= 0).sum())
print('Zero/negative price rows remaining:', (df['UnitPrice'] <= 0).sum())

df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

print('Remaining rows:', len(df))

Negative quantity rows remaining: 1336
Zero/negative price rows remaining: 2517
Remaining rows: 530104


## Step 4: Handle missing values

- **Missing Description:** drop these rows — without a product name we can't analyze the product, and it's a small share of data.
- **Missing CustomerID:** ~25% of rows. We will NOT drop all of these, since they still represent real revenue. Instead, we'll keep them for product/revenue/country analysis, but exclude them from customer-level analysis (top customers, repeat buyers, etc.) since we can't identify the customer.

In [7]:
df = df.dropna(subset=['Description'])
print('Rows after dropping missing descriptions:', len(df))

# Flag missing customers instead of dropping — keep for revenue analysis
df['HasCustomerID'] = df['CustomerID'].notnull()
print('Rows with known customer:', df['HasCustomerID'].sum())
print('Rows with unknown customer:', (~df['HasCustomerID']).sum())

Rows after dropping missing descriptions: 530104
Rows with known customer: 397884
Rows with unknown customer: 132220


## Step 5: Remove duplicates

In [8]:
print('Duplicates before:', df.duplicated().sum())
df = df.drop_duplicates()
print('Rows after removing duplicates:', len(df))

Duplicates before: 5226
Rows after removing duplicates: 524878


## Step 6: Fix data types

In [9]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df['CustomerID'] = df['CustomerID'].astype('Int64')  # nullable integer type

df['StockCode'] = df['StockCode'].astype(str)
df['Description'] = df['Description'].astype(str).str.strip()
df['Country'] = df['Country'].astype(str).str.strip()

df.dtypes

InvoiceNo                   str
StockCode                   str
Description                 str
Quantity                  int64
InvoiceDate      datetime64[us]
UnitPrice               float64
CustomerID                Int64
Country                     str
HasCustomerID              bool
dtype: object

## Step 7: Feature engineering

Create columns that will make our business questions easier to answer later in SQL and Excel.

In [10]:
# Revenue per line item
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Date parts for trend analysis
df['InvoiceYear'] = df['InvoiceDate'].dt.year
df['InvoiceMonth'] = df['InvoiceDate'].dt.month
df['InvoiceYearMonth'] = df['InvoiceDate'].dt.to_period('M').astype(str)
df['InvoiceDay'] = df['InvoiceDate'].dt.day_name()
df['InvoiceHour'] = df['InvoiceDate'].dt.hour

df[['InvoiceDate','Quantity','UnitPrice','Revenue','InvoiceYearMonth','InvoiceDay']].head()

,InvoiceDate,Quantity,UnitPrice,Revenue,InvoiceYearMonth,InvoiceDay
0,2010-12-01 08:26:00,6,2.55,15.30,2010-12,Wednesday
1,2010-12-01 08:26:00,6,3.39,20.34,2010-12,Wednesday
2,2010-12-01 08:26:00,8,2.75,22.00,2010-12,Wednesday
3,2010-12-01 08:26:00,6,3.39,20.34,2010-12,Wednesday
4,2010-12-01 08:26:00,6,3.39,20.34,2010-12,Wednesday


## Step 8: Final check and export

In [11]:
print('Final shape:', df.shape)
print()
print('Remaining nulls:')
print(df.isnull().sum())
print()
print('Date range:', df['InvoiceDate'].min(), 'to', df['InvoiceDate'].max())
print('Total revenue: £{:,.2f}'.format(df['Revenue'].sum()))

Final shape: (524878, 15)

Remaining nulls:
InvoiceNo                0
StockCode                0
Description              0
Quantity                 0
InvoiceDate              0
UnitPrice                0
CustomerID          132186
Country                  0
HasCustomerID            0
Revenue                  0
InvoiceYear              0
InvoiceMonth             0
InvoiceYearMonth         0
InvoiceDay               0
InvoiceHour              0
dtype: int64

Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00
Total revenue: £10,642,110.80


In [12]:
# Export cleaned data — this is what we'll load into SQL and use in Excel
df.to_csv('../data/online_retail_cleaned.csv', index=False)
print('Saved cleaned dataset.')

Saved cleaned dataset.
